In [ ]:
# Default values for parameters. Papermill will inject overrides from run_experiments.py.
ALGORITHM = 'block_wanda'
BLOCK_ROWS = 1
BLOCK_COLS = 2
MAX_ITER = 1
RANDOM_SWAPS = 10
INNER_REFINE = 2
SPARSITY = 0.5
SWAP_FRACTION = 1/30
SORT_START = True
SEED = 47
SKIP_PERPLEXITY = False
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
import time
import sys
sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.nn.functional as F

from modelutils import *
from datautils import *

set_seed(SEED)

In [ ]:
def get_opt(model):
    import torch
    def skip(*args, **kwargs):
        pass
    torch.nn.init.kaiming_uniform_ = skip
    torch.nn.init.uniform_ = skip
    torch.nn.init.normal_ = skip
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(model, torch_dtype='auto', cache_dir="llm_weights")
    print("ms", model.config.max_position_embeddings)
    model.seqlen = model.config.max_position_embeddings
    return model

In [ ]:
# Create model

model_name = MODEL_NAME

model = get_opt(model_name)
model.eval()

def _safe_name(s):
    return s.replace("/", "_").replace(":", "_")

MODEL_NAME_SAFE = _safe_name(model_name)
OUTPUT_FOLDER = f"benchmark_csvs_smollm_{MODEL_NAME_SAFE}"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
n_samples = 32

dataloader, testloader = get_loaders(
    "c4", nsamples=n_samples, seed=0, model=model_name, seqlen=model.seqlen
)

In [ ]:
from tetris import tetris_pruning, block_sparsity_pruning, sort_columns_by_norm, original_tetris_pruning, random_swaps

def do_block_wanda_sparsity_per_row(lx):
    # 1. Calculate Wanda scores
    norm = lx.input_sq_norms.sqrt() + 1e-8
    scores = (lx.weight.float().detach() * norm).square()
    
    rows, cols = scores.shape
    block_rows, block_cols = BLOCK_ROWS, BLOCK_COLS

    n_blocks_row = rows // block_rows
    n_blocks_col = cols // block_cols

    # 2. Reshape into blocks and sum the Wanda scores
    cropped_scores = scores[:n_blocks_row*block_rows, :n_blocks_col*block_cols]
    blocks = cropped_scores.reshape(n_blocks_row, block_rows, n_blocks_col, block_cols)
    block_norms = blocks.sum(dim=(1, 3))

    # 3. Find the exact column index to cut off the bottom sparsity %
    k = int(n_blocks_col * SPARSITY)
    
    # Sort each row individually, and grab the threshold for that specific row
    threshold = block_norms.sort(dim=1)[0][:, k]

    # 4. Create block mask and expand to original size
    block_mask = block_norms >= threshold.unsqueeze(1)
    mask = block_mask.repeat_interleave(block_rows, dim=0).repeat_interleave(block_cols, dim=1)

    # 5. Handle padding for remainders
    pad_bottom = rows - mask.shape[0]
    pad_right = cols - mask.shape[1]
    
    if pad_bottom > 0 or pad_right > 0:
        # F is torch.nn.functional
        mask = F.pad(mask, (0, pad_right, 0, pad_bottom), value=False)

    # 6. Apply mask directly to original weights
    mask = mask.to(lx.weight.device)
    W_pruned = lx.weight * mask

    return W_pruned

def do_original_tetris_block_wanda(lx):
    # 1. Extract weights and calculate wanda scores
    norm = lx.input_sq_norms.sqrt() + 1e-8
    wanda_scores = (lx.weight.float().detach() * norm).square()

    # 2. Run original Tetris on the wanda scores
    _, _, perm, _, _ = original_tetris_pruning(
        W=wanda_scores, 
        block_size=(BLOCK_ROWS, BLOCK_COLS),
        sparsity=SPARSITY,
        max_iter=MAX_ITER,
        verbose=False,
    )

    permuted_scores = wanda_scores[:, perm]
    _, best_score_mask = block_sparsity_pruning(permuted_scores, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY)
    
    # 3. Apply permutation and mask
    W_permuted = lx.weight[:, perm]
    W_pruned = W_permuted * best_score_mask
    
    # 4. Un-permute back to the original input order for safe evaluation
    inverse_perm = torch.argsort(perm)
    W_final_unpermuted = W_pruned[:, inverse_perm]
    
    # 5. Return the final tensor
    return W_final_unpermuted


def do_sort_columns_by_norm_block_wanda(lx):
    # 1. Extract weights and calculate wanda scores
    norm = lx.input_sq_norms.sqrt() + 1e-8
    wanda_scores = (lx.weight.float().detach() * norm).square()

    # 2. Run pruning algorithm
    _, mask, perm = sort_columns_by_norm(
        W=wanda_scores,
        block_size=(BLOCK_ROWS, BLOCK_COLS),
        sparsity=SPARSITY,
        verbose=False
    )

    # 3. Apply mask
    W_pruned = lx.weight[:, perm] * mask
    
    # 4. Un-permute back to the original input order for safe evaluation
    inverse_perm = torch.argsort(perm)
    W_final_unpermuted = W_pruned[:, inverse_perm]

    # 5. Return the final tensor
    return W_final_unpermuted

def do_random_swaps_block_wanda(lx):
    # 1. Extract weights and calculate wanda scores
    norm = lx.input_sq_norms.sqrt() + 1e-8
    wanda_scores = (lx.weight.float().detach() * norm).square()


    # W_current, mask, permutation, best_time_relative, history
    # 2. Run pruning algorithm
    _, mask, perm, _, _ = random_swaps(
        W=wanda_scores,
        max_iter=MAX_ITER,
        block_size=(BLOCK_ROWS, BLOCK_COLS),
        sparsity=SPARSITY,
        sort_start=SORT_START,
        swap_fraction=SWAP_FRACTION,
        verbose=False,
    )

    # 3. Apply mask
    W_pruned = lx.weight[:, perm] * mask
    
    # 4. Un-permute back to the original input order for safe evaluation
    inverse_perm = torch.argsort(perm)
    W_final_unpermuted = W_pruned[:, inverse_perm]

    # 5. Return the final tensor
    return W_final_unpermuted


def do_tetris_block_wanda(lx):
    # 1. Extract weights and calculate wanda scores
    norm = lx.input_sq_norms.sqrt() + 1e-8
    wanda_scores = (lx.weight.float().detach() * norm).square()
    
    # 2. Run Tetris on the wanda scores
    _, _, perm, _, _, _ = tetris_pruning(
        W=wanda_scores,
        block_size=(BLOCK_ROWS, BLOCK_COLS),
        sparsity=SPARSITY,
        max_iter=MAX_ITER,
        random_swaps=RANDOM_SWAPS,
        inner_refine=INNER_REFINE,
        verbose=False
    )

    permuted_scores = wanda_scores[:, perm]
    _, best_score_mask = block_sparsity_pruning(permuted_scores, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY)
    
    # 3. Apply permutation and mask
    W_permuted = lx.weight[:, perm]
    W_pruned = W_permuted * best_score_mask
    
    # 4. Un-permute back to the original input order for safe evaluation
    inverse_perm = torch.argsort(perm)
    W_final_unpermuted = W_pruned[:, inverse_perm]
    
    # 5. Return the final tensor
    return W_final_unpermuted

In [ ]:
@torch.no_grad()
def opt_sequential(model, dataloader, dev):
    print('Starting ...')

    use_cache = model.config.use_cache
    model.config.use_cache = False
    layers = model.model.layers

    model.model.embed_tokens = model.model.embed_tokens.to(dev) 
    model.model.rotary_emb = model.model.rotary_emb.to(dev)
    layers[0] = layers[0].to(dev)

    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros(
        (n_samples, model.seqlen, model.config.hidden_size), dtype=dtype, device=dev
    )
    cache = {'i': 0, 'attention_mask': None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            cache['position_embeddings'] = kwargs['position_embeddings']
            raise ValueError
    layers[0] = Catcher(layers[0])
    for batch in dataloader:
        try:
            model(batch[0].to(dev))
        except ValueError:
            pass
    layers[0] = layers[0].module

    layers[0] = layers[0].cpu()
    model.model.embed_tokens = model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()

    outs = torch.zeros_like(inps)
    attention_mask = cache['attention_mask']
    position_embeddings = cache['position_embeddings']

    print('Ready.')

    errors = []
    layer_metrics = []
    pruned_weights = {}
    # Collect input norm and prune after
    for i in range(len(layers)):
        layer = layers[i].to(dev)

        subset = find_layers(layer)

        def add_batch(name):
            def tmp(layer, inp, out):
                X = inp[0].detach().float()
                X = X.reshape(-1, X.shape[-1])
                layer.input_sq_norms += X.square().mean(dim=0)
            return tmp
        handles = []
        for name in subset:
            subset[name].batches = []
            subset[name].input_sq_norms = torch.zeros(subset[name].weight.shape[1], device=subset[name].weight.device)        
            handles.append(subset[name].register_forward_hook(add_batch(name)))
        for j in range(n_samples):
            outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask, position_embeddings=position_embeddings)[0]
        for h in handles:
            h.remove()

        for name in subset:
            print(i, name)
            print('Pruning ...')
            lx = subset[name]
            
            s1 = time.time()

            torch.cuda.synchronize()
            start_time = time.perf_counter()
            
            match ALGORITHM:
                case 'block_wanda':
                    W2 = do_block_wanda_sparsity_per_row(lx)
                case 'our_tetris':
                    W2 = do_tetris_block_wanda(lx)
                case 'original_tetris':
                    W2 = do_original_tetris_block_wanda(lx)
                case 'sort_columns_by_norm':
                    W2 = do_sort_columns_by_norm_block_wanda(lx)
                case 'random_swaps':
                    W2 = do_random_swaps_block_wanda(lx)
                case _:
                    raise ValueError(f"Unknown algorithm: {ALGORITHM}")
                
            torch.cuda.synchronize()
            total_time = time.perf_counter() - start_time
            
            # relative error
            print(W2.shape, lx.weight.shape)
            rel_error = ((lx.weight - W2).float().square() * lx.input_sq_norms).sum().item() / (lx.weight.float().square() * lx.input_sq_norms).sum().item()
            density = (W2 != 0).sum().item() / W2.numel()
            errors.append(rel_error)
            print("rel err", rel_error, "density", density)

            # Score-space improvement vs block-Wanda baseline
            norm = lx.input_sq_norms.sqrt() + 1e-8
            score = (lx.weight.float().detach() * norm).square()

            # Baseline: plain block pruning, no permutation
            _, baseline_mask = block_sparsity_pruning(
                score, block_size=(BLOCK_ROWS, BLOCK_COLS), sparsity=SPARSITY,
            )
            score_baseline_sum = score[baseline_mask == 0].sum().item()

            algo_pruned_positions = (W2 == 0)
            score_pruned_sum = score[algo_pruned_positions].sum().item()

            if score_baseline_sum > 0:
                score_improvement_pct = 100 * (score_baseline_sum - score_pruned_sum) / score_baseline_sum
            else:
                score_improvement_pct = 0.0

            # wanda weighted masses
            retained_wanda_mass = (W2.float().square() * lx.input_sq_norms).sum().item()

            # ablsotute pruned weight sum
            pruned_weight_l1 = (lx.weight - W2).abs().sum().item()
            total_weight_l1  = lx.weight.abs().sum().item()
        

            # Append everything to the tracker
            layer_metrics.append({
                "layer_name": f"blocks.{i}.{name}",       
                "algorithm": ALGORITHM,
                "block_rows": BLOCK_ROWS,
                "block_cols": BLOCK_COLS,
                "sparsity": SPARSITY,
                "max_iter": MAX_ITER,
                "random_swaps": RANDOM_SWAPS,
                "swap_fraction": SWAP_FRACTION,
                "sort_start": SORT_START,
                "total_time_sec": total_time,
                "rel_error": rel_error,
                "density": density,
                "pruned_weight_l1": pruned_weight_l1,
                "total_weight_l1": total_weight_l1,
                "total_wanda_mass": (lx.weight.float().square() * lx.input_sq_norms).sum().item(),
                "retained_wanda_mass": retained_wanda_mass,
                "score_pruned_sum": score_pruned_sum,
                "score_baseline_sum": score_baseline_sum,
                "score_improvement_pct": score_improvement_pct,
                "n_rows": W2.shape[0],
                "n_cols": W2.shape[1],
                "perplexity_baseline": float("nan"),       # filled in cell 10
                "perplexity_all_layers": float("nan"),     # filled in cell 10
            })
            
            pruned_weights[(i, name)] = W2.detach().clone() # dont overwrite weights, just store
        
            

        for j in range(n_samples):
            outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask, position_embeddings=position_embeddings)[0]
        
        

        layers[i] = layer.cpu()
        del layer
        
        
        
        torch.cuda.empty_cache()

        inps, outs = outs, inps

    # Overwrite weights after all pruning is done
    for i, layer in enumerate(layers):
        layer = layer.to(dev)
        subset = find_layers(layer)
        for name, lx in subset.items():
            if (i, name) in pruned_weights:
                lx.weight.data = pruned_weights[(i, name)].to(lx.weight)
        layers[i] = layer.cpu()
        torch.cuda.empty_cache()

    model.config.use_cache = use_cache
    return errors, layer_metrics

In [ ]:
errors = []
layer_metrics = []
execution_time = 0

if ALGORITHM != 'no_prune':
    start = time.time()
    errors, layer_metrics = opt_sequential(model, dataloader, DEV)
    execution_time = time.time() - start
    print("total time", execution_time)

In [ ]:
@torch.no_grad()
def opt_eval(model, testenc, dev, dataset: str, log_wandb: bool = False):
    print('Evaluating ...')

    testenc = testenc.input_ids
    nsamples = testenc.numel() // model.seqlen

    use_cache = model.config.use_cache
    model.config.use_cache = False
    layers = model.model.layers

    model.model.embed_tokens = model.model.embed_tokens.to(dev)
    model.model.rotary_emb = model.model.rotary_emb.to(dev)
    layers[0] = layers[0].to(dev)

    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros(
        (nsamples, model.seqlen, model.config.hidden_size), dtype=dtype, device=dev
    )
    cache = {'i': 0, 'attention_mask': None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            cache['position_embeddings'] = kwargs['position_embeddings']
            raise ValueError
    layers[0] = Catcher(layers[0])
    for i in range(nsamples):
        batch = testenc[:, (i * model.seqlen):((i + 1) * model.seqlen)].to(dev)
        try:
            model(batch)
        except ValueError:
            pass
    layers[0] = layers[0].module

    layers[0] = layers[0].cpu()
    model.model.embed_tokens = model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()

    outs = torch.zeros_like(inps)
    attention_mask = cache['attention_mask']
    position_embeddings = cache['position_embeddings']

    for i in range(len(layers)):
        print(i)
        layer = layers[i].to(dev)

        
        for j in range(nsamples):
            outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask, position_embeddings=position_embeddings)[0]
        layers[i] = layer.cpu()
        del layer
        torch.cuda.empty_cache()
        inps, outs = outs, inps

    if model.model.norm is not None:
        model.model.norm = model.model.norm.to(dev)
    model.lm_head = model.lm_head.to(dev)

    testenc = testenc.to(dev)
    nlls = []
    for i in range(nsamples):
        hidden_states = inps[i].unsqueeze(0)
        if model.model.norm is not None:
            hidden_states = model.model.norm(hidden_states)
        lm_logits = model.lm_head(hidden_states)
        shift_logits = lm_logits[:, :-1, :].contiguous()
        shift_labels = testenc[
            :, (i * model.seqlen):((i + 1) * model.seqlen)
        ][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * model.seqlen
        nlls.append(neg_log_likelihood)
    ppl = torch.exp(torch.stack(nlls).sum() / (nsamples * model.seqlen))
    print(f"Perplexity: {ppl.item():3f}")
    if log_wandb:
         wandb.log({f'{dataset}/perplexity': ppl.item()})

    model.config.use_cache = use_cache

    return ppl.item()

In [ ]:
whole_model_eval_time = float("nan")

# Skip perplexity if requested
if SKIP_PERPLEXITY:
    final_ppl = float("nan")
    baseline_ppl = float("nan")
    print("Skipping perplexity (SKIP_PERPLEXITY=True)")
else:
    BASELINE_PPL_CACHE = f"baseline_ppl_{MODEL_NAME_SAFE}.pt"
    if os.path.exists(BASELINE_PPL_CACHE):
        baseline_ppl = torch.load(BASELINE_PPL_CACHE)["baseline_ppl"]
        print(f"Loaded cached baseline perplexity: {baseline_ppl:.4f}")
    else:
        # Reload an unpruned copy of the model for baseline eval
        # (the model in scope has been pruned in-place by opt_sequential)
        unpruned = get_opt(model_name).eval()
        for dataset in ["wikitext2"]:
            _, testloader_baseline = get_loaders(dataset, seed=0, model=model_name, seqlen=unpruned.seqlen)
            baseline_ppl = opt_eval(unpruned, testloader_baseline, DEV, dataset, False)
        torch.save({"model_name": model_name, "baseline_ppl": baseline_ppl}, BASELINE_PPL_CACHE)
        print(f"Cached baseline perplexity {baseline_ppl:.4f} to {BASELINE_PPL_CACHE}")
        del unpruned
        torch.cuda.empty_cache()

    # Pruned-model perplexity
    t0 = time.perf_counter()
    for dataset in ["wikitext2"]:
        dataloader, testloader = get_loaders(dataset, seed=0, model=model_name, seqlen=model.seqlen)
        print(dataset)
        final_ppl = opt_eval(model, testloader, DEV, dataset, False)
    whole_model_eval_time = time.perf_counter() - t0
    print(f"Whole-model evaluation time: {whole_model_eval_time:.2f} seconds")

# Patch model-level perplexity into every row of layer_metrics, then save
for row in layer_metrics:
    row["perplexity_baseline"] = baseline_ppl
    row["perplexity_all_layers"] = final_ppl

import scrapbook as sb
import pandas as pd

csv_name = (
    f"layer_metrics_{ALGORITHM}"
    f"_{BLOCK_ROWS}x{BLOCK_COLS}"
    f"_iters_{MAX_ITER}"
    f"_swaps_{RANDOM_SWAPS}"
    f"_innerref_{INNER_REFINE}"
    f"_sortstart_{SORT_START}"
    f"_sparsity_{SPARSITY}.csv"
)
csv_path = os.path.join(OUTPUT_FOLDER, csv_name)
df_metrics = pd.DataFrame(layer_metrics)
df_metrics.to_csv(csv_path, index=False)
print(f"Wrote {csv_path}")

sb.glue("perplexity", final_ppl)
sb.glue("baseline_perplexity", baseline_ppl)
sb.glue("execution_time", execution_time)
sb.glue("relative_errors", errors)
sb.glue("layer_metrics_csv", csv_path)
sb.glue("whole_model_eval_time_sec", whole_model_eval_time)